# Jumper: a GPU training session in Google Colab

Choose **Runtime → Change runtime type → GPU**, then run the cells in order. A paid runtime is not required by this notebook; GPU availability, model and session duration are decided by Colab. No GPU means a clear stop, never a silent CPU training run.

The session uses the repository's existing training, replay and ONNX export commands in an isolated `.venv`. The Colab kernel only coordinates subprocesses and displays results. First watch a bundled pretrained dance, then run a five-iteration smoke test and train a modest walking run. A short training run is a workflow check, not a promise of a learned gait or a hardware-ready robot.

Optional Drive snapshots happen **during training**, each time a complete checkpoint is saved. Without Drive, download the final backup before Colab discards `/content`. Snapshots retain a checkpoint with its source/runtime manifest; writes still require time and an abrupt VM loss can interrupt an unfinished copy.

This notebook checks the **training runtime**. It does not set up Rust, RKNN, DDS or the physical motor bus. An ONNX export is not a robot deployment bundle.

## 1. Session choices

The defaults select the accompanying release branch in the tianrking/jumper fork. After the change is merged, you can select KingKongRobotics/jumper and main instead. When testing another fork or release branch, enter its exact repository URL and revision. The requested revision is resolved to one commit and recorded; an existing checkout is reused only if its origin and current commit agree. It is never reset or replaced.

The source revision must contain `mjrl.colab` and the replay video options. Setup stops early if you selected older code. For later reproducible sessions, paste the printed 40-character commit into `SOURCE_REVISION`.

Select a single explicit saved checkpoint when resuming. Restore a downloaded ZIP in section 5, or enter a checkpoint inside a mounted Drive snapshot. Resume refuses changes to the task, model, backend, number of environments, source content or core runtime package versions.

The replay form controls duration, frame rate, image size, scene and camera distance. **task default** keeps the task's world; other scenes can change ground friction or terrain and test a different condition. The bundled dance stays at its recorded ten-second preview settings. Set **DOWNLOAD_FILES** off to run all cells without automatic browser downloads; the artifacts and embedded previews remain available.

For a later resume, set **SOURCE_REVISION** to the recorded commit and paste the backup manifest's `config.runtime_versions` JSON into **CORE_VERSION_OVERRIDES**. Only six named core packages and exact versions are accepted. The tested Torch 2.9.1 / torchvision 0.24.1 pair is fixed; a different Torch pair requires a separately validated notebook. A saved CUDA wheel still needs a compatible GPU allocation.

In [ ]:
import json
import math
import re

REPOSITORY_URL = "https://github.com/tianrking/jumper.git"  # @param {type:"string"}
SOURCE_REVISION = "codex/colab-training-and-video"  # @param {type:"string"}
CHECKOUT_FOLDER = "jumper"  # @param {type:"string"}
TASK = "jumper.tripod"  # @param ["jumper.flat", "jumper.tripod", "jumper.tetrapod", "jumper.ripple", "jumper.posture", "jumper.five_foot", "jumper.dance", "jumper.jump", "jumper.ref_free_jump", "jumper.swing"]
MODEL = "jumper"
NUM_ENVS = 256  # @param {type:"integer"}
ITERATIONS = 500  # @param {type:"integer"}
SIM_SECONDS = 10.0  # @param {type:"number"}
SIM_FPS = 30.0  # @param {type:"number"}
SIM_WIDTH = 640  # @param {type:"integer"}
SIM_HEIGHT = 480  # @param {type:"integer"}
SIM_SCENE = "task default"  # @param ["task default", "default", "studio", "daylight", "beach", "football", "soft", "ice", "rubber", "plain", "rough"]
SIM_CAMERA_DISTANCE = 0.0  # @param {type:"number"}
RECORD_JOINTS = True  # @param {type:"boolean"}
DOWNLOAD_FILES = True  # @param {type:"boolean"}
USE_DRIVE = False  # @param {type:"boolean"}
UPLOAD_BACKUP = False  # @param {type:"boolean"}
RESUME_CHECKPOINT = ""  # @param {type:"string"}
CORE_VERSION_OVERRIDES = ""  # @param {type:"string"}
DRIVE_FOLDER = "JumperColab"  # @param {type:"string"}

# Validate forms before attaching subprocesses or downloading dependencies.
if not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_-]{0,63}", CHECKOUT_FOLDER):
    raise ValueError("CHECKOUT_FOLDER must be one simple folder name under /content")
if not isinstance(NUM_ENVS, int) or not isinstance(ITERATIONS, int) or min(NUM_ENVS, ITERATIONS) < 1:
    raise ValueError("NUM_ENVS and ITERATIONS must be positive integers")
if not SOURCE_REVISION.strip() or SOURCE_REVISION.startswith("-"):
    raise ValueError("Enter a branch, tag or commit as SOURCE_REVISION")
if any(not math.isfinite(value) or value <= 0 for value in (SIM_SECONDS, SIM_FPS)):
    raise ValueError("SIM_SECONDS and SIM_FPS must be finite and positive")
if any(not isinstance(value, int) or value < 2 or value % 2 for value in (SIM_WIDTH, SIM_HEIGHT)):
    raise ValueError("SIM_WIDTH and SIM_HEIGHT must be even integers of at least 2")
if not math.isfinite(SIM_CAMERA_DISTANCE) or SIM_CAMERA_DISTANCE < 0:
    raise ValueError("SIM_CAMERA_DISTANCE must be zero for automatic framing or positive")
if SIM_SCENE not in (
    "task default", "default", "studio", "daylight", "beach", "football",
    "soft", "ice", "rubber", "plain", "rough",
):
    raise ValueError("Choose an available replay scene from the form")
CORE_PACKAGES = {"torch", "mujoco", "mujoco-warp", "warp-lang", "numpy", "tensordict"}
CORE_VERSIONS = json.loads(CORE_VERSION_OVERRIDES) if CORE_VERSION_OVERRIDES.strip() else {}
if not isinstance(CORE_VERSIONS, dict) or set(CORE_VERSIONS) - CORE_PACKAGES:
    raise ValueError("CORE_VERSION_OVERRIDES must be a JSON object of recorded core package versions")
if any(not isinstance(version, str) or not re.fullmatch(
    r"[0-9]+(?:\.[0-9]+)+(?:[a-zA-Z]+[0-9]*)?(?:\+[A-Za-z0-9_.-]+)?", version
) for version in CORE_VERSIONS.values()):
    raise ValueError("Core version overrides must contain exact numeric package versions")
if "torch" in CORE_VERSIONS and CORE_VERSIONS["torch"].split("+")[0] != "2.9.1":
    raise ValueError("This notebook supports the tested torch 2.9.1 / torchvision 0.24.1 pair only")

## 2. Resolve the checkout and create an isolated interpreter

In [ ]:
import json
import os
import signal
import subprocess
import sys
import uuid
import venv
from collections import deque
from pathlib import Path

if not (3, 10) <= sys.version_info[:2] <= (3, 13):
    raise RuntimeError("Jumper supports Python 3.10–3.13; select a compatible Colab runtime")
if not Path("/content").is_dir():
    raise RuntimeError("This notebook is intended for Google Colab with /content available")

def run(command, *, cwd=None):
    print("+", " ".join(str(part) for part in command), flush=True)
    command = [str(part) for part in command]
    tail = deque(maxlen=60)
    process = subprocess.Popen(
        command, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1, start_new_session=True,
    )
    try:
        for line in process.stdout:
            print(line, end="", flush=True)
            tail.append(line[-4096:])
        returncode = process.wait()
        output = "".join(tail)
        if returncode:
            raise subprocess.CalledProcessError(returncode, command, output=output)
        return subprocess.CompletedProcess(command, returncode, stdout=output)
    except BaseException:
        if process.poll() is None:
            # Interrupt the whole subprocess group so helper cleanup can save checkpoints.
            try:
                os.killpg(process.pid, signal.SIGINT)
            except ProcessLookupError:
                pass
            try:
                process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                try:
                    os.killpg(process.pid, signal.SIGTERM)
                except ProcessLookupError:
                    pass
                try:
                    process.wait(timeout=10)
                except subprocess.TimeoutExpired:
                    os.killpg(process.pid, signal.SIGKILL)
                    process.wait()
        raise
    finally:
        process.stdout.close()

# Fail before downloads when no GPU was attached to this runtime.
run(["nvidia-smi", "-L"])
gpu_rows = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=name,compute_cap", "--format=csv,noheader"],
    text=True,
).strip().splitlines()
if not gpu_rows:
    raise RuntimeError("No GPU detected; change the Colab runtime type to GPU")
gpu_name, capability = [part.strip() for part in gpu_rows[0].rsplit(",", 1)]
capability = float(capability)
if capability < 7.5:
    raise RuntimeError(f"Unsupported GPU for this notebook's tested wheel family: {gpu_name}")
print("GPU:", gpu_name, "compute capability:", capability)

REPO = Path("/content") / CHECKOUT_FOLDER
created_checkout = not REPO.exists()
if created_checkout:
    run(["git", "clone", REPOSITORY_URL, REPO])
else:
    if not (REPO / ".git").exists():
        raise RuntimeError(f"{REPO} already exists and is not a Git checkout")
    origin = subprocess.check_output(
        ["git", "remote", "get-url", "origin"], cwd=REPO, text=True,
    ).strip()
    if origin.rstrip("/").removesuffix(".git") != REPOSITORY_URL.rstrip("/").removesuffix(".git"):
        raise RuntimeError(f"Existing checkout origin differs: {origin}")

# Fetching updates references; no checkout content is overwritten.
run(["git", "fetch", "origin", SOURCE_REVISION], cwd=REPO)
COMMIT = subprocess.check_output(
    ["git", "rev-parse", "FETCH_HEAD^{commit}"], cwd=REPO, text=True,
).strip()
head = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip()
if head != COMMIT:
    # A freshly cloned clean checkout may be pinned. Reused checkouts are never switched.
    status = subprocess.check_output(["git", "status", "--porcelain"], cwd=REPO, text=True)
    if not created_checkout or status:
        raise RuntimeError(f"Existing checkout is {head}, requested {COMMIT}; use a new runtime")
    run(["git", "checkout", "--detach", COMMIT], cwd=REPO)
if not (REPO / "rl/mjrl/colab.py").is_file():
    raise RuntimeError(
        f"Revision {COMMIT} does not contain this notebook's Colab helper. "
        "Choose the accompanying release/fork revision, then rerun in a fresh runtime."
    )
print("Pinned source commit:", COMMIT)

PYTHON = REPO / ".venv/bin/python"
if not PYTHON.exists():
    venv.EnvBuilder(with_pip=False, system_site_packages=False).create(REPO / ".venv")
run([PYTHON, "-c", "import sys; print('isolated interpreter:', sys.executable, sys.prefix)"])
# Colab can omit ensurepip. Host pip manages only the explicitly selected venv.
# This also repairs a previously created interpreter that has no pip yet.
kernel_pip_version = subprocess.check_output(
    [sys.executable, "-m", "pip", "--version"], text=True,
).split()[1]
if tuple(int(part) for part in kernel_pip_version.split(".")[:2]) < (22, 3):
    raise RuntimeError("The Colab kernel needs pip 22.3+ for --python; select a newer runtime")
run([sys.executable, "-m", "pip", "--python", PYTHON, "install", "--upgrade", "pip"])
SESSION = Path("/content/jumper-colab") / uuid.uuid4().hex
SESSION.mkdir(parents=True)
os.environ["MUJOCO_GL"] = "egl"
os.environ["PYTHONUNBUFFERED"] = "1"
os.environ["MPLBACKEND"] = "Agg"

## 3. Install and verify the training runtime

Dependencies are installed only into this checkout's `.venv`. The environment is created without ensurepip; the kernel's pip 22.3+ uses its [official --python option](https://pip.pypa.io/en/stable/topics/python-option/) to bootstrap pip only into the target environment, including a half-created venv. Kernel packages are not changed. The notebook pins the paired PyTorch 2.9.1 / torchvision 0.24.1 wheels from the [official PyTorch archive](https://pytorch.org/get-started/previous-versions/): CUDA 12.6 for a T4, CUDA 12.8 for Ampere/Ada/Blackwell. A real CUDA operation and a compiled Warp CUDA kernel must succeed; package installation alone is not acceptance.

Other dependencies follow `pyproject.toml`; the exact core versions are saved for resume checks. A driver or wheel mismatch raises an error. Do not continue past a failed gate.

A session-local GLVND configuration selects an existing NVIDIA EGL library. A fresh isolated child must report an actual NVIDIA GL vendor and render nonempty 64×64 RGB pixels before the CUDA/Warp gates and task commands continue. The result is saved in graphics.json; system vendor files and GPU drivers are not replaced. This prevents a GPU training session from silently encoding previews with Mesa software rendering.

In [ ]:
run(["apt-get", "update", "-qq"])
run(["apt-get", "install", "-y", "-qq", "libegl1", "libgl1", "ffmpeg"])
run([PYTHON, "-m", "pip", "install", "--upgrade", "pip"])
TORCH_INDEX = "cu126" if capability < 8.0 else "cu128"
requested_torch = CORE_VERSIONS.get("torch", "2.9.1")
if "+" in requested_torch and requested_torch != f"2.9.1+{TORCH_INDEX}":
    raise RuntimeError(
        f"Saved Torch wheel {requested_torch} needs a compatible GPU; this GPU selects {TORCH_INDEX}"
    )
run([
    PYTHON, "-m", "pip", "install", "torch==2.9.1", "torchvision==0.24.1",
    "--index-url", f"https://download.pytorch.org/whl/{TORCH_INDEX}",
])
run([PYTHON, "-m", "pip", "install", "-e", str(REPO)])
# Restore only explicitly allowed exact versions; no shell, URLs or pip flags are accepted.
core_requirements = [f"{name}=={version}" for name, version in sorted(CORE_VERSIONS.items())
                     if name != "torch"]
if core_requirements:
    run([PYTHON, "-m", "pip", "install", *core_requirements])
run([PYTHON, "-m", "pip", "install", "matplotlib"])
run([PYTHON, "-m", "pip", "check"])

# Colab may expose NVIDIA's driver without registering its EGL vendor with GLVND.
# Keep the override in this session; never replace a system vendor configuration.
def configure_nvidia_egl(session, library_candidates):
    library = next((
        Path(candidate).resolve() for candidate in library_candidates
        if Path(candidate).name.startswith("libEGL_nvidia.so.") and Path(candidate).is_file()
    ), None)
    if library is None:
        raise RuntimeError(
            "No preinstalled NVIDIA EGL library was found in the standard driver paths. "
            "Select a compatible GPU runtime; this notebook will not use software rendering."
        )
    vendor_config = Path(session) / "nvidia-egl.json"
    vendor_config.write_text(json.dumps({
        "file_format_version": "1.0.0", "ICD": {"library_path": str(library)},
    }, indent=2) + "\n", encoding="utf-8")
    os.environ["__EGL_VENDOR_LIBRARY_FILENAMES"] = str(vendor_config.resolve())
    driver_directory = str(library.parent)
    existing = [
        entry for entry in os.environ.get("LD_LIBRARY_PATH", "").split(":")
        if entry and entry != driver_directory
    ]
    os.environ["LD_LIBRARY_PATH"] = ":".join([driver_directory, *existing])
    os.environ["MUJOCO_GL"] = "egl"
    os.environ["PYOPENGL_PLATFORM"] = "egl"
    return library

NVIDIA_EGL = configure_nvidia_egl(SESSION, (
    "/usr/lib64-nvidia/libEGL_nvidia.so.0",
    "/usr/lib/x86_64-linux-gnu/libEGL_nvidia.so.0",
    "/usr/lib/x86_64-linux-gnu/nvidia/current/libEGL_nvidia.so.0",
    "/usr/lib/nvidia/libEGL_nvidia.so.0",
))
print("Session NVIDIA EGL library:", NVIDIA_EGL)

# A fresh isolated child must verify the actual GL vendor, not just library presence.
graphics_probe = r"""import json
import os
from pathlib import Path
import sys
import time
import mujoco
import numpy as np
from OpenGL import GL

model = mujoco.MjModel.from_xml_string(
    '<mujoco><visual><global offwidth="64" offheight="64"/></visual>'
    '<worldbody><light pos="0 0 3"/>'
    '<geom type="plane" size="1 1 .1" rgba=".3 .3 .3 1"/>'
    '<geom type="box" pos="0 0 .12" size=".1 .1 .12" rgba=".8 .2 .1 1"/>'
    '</worldbody></mujoco>'
)
data = mujoco.MjData(model)
mujoco.mj_forward(model, data)
renderer = mujoco.Renderer(model, height=64, width=64)
try:
    renderer.update_scene(data)
    started = time.perf_counter()
    pixels = renderer.render()
    render_ms = (time.perf_counter() - started) * 1000
    vendor = (GL.glGetString(GL.GL_VENDOR) or b"").decode("utf-8", "replace")
    device = (GL.glGetString(GL.GL_RENDERER) or b"").decode("utf-8", "replace")
    nonzero_pixels = int(np.count_nonzero(np.any(pixels != 0, axis=-1)))
    if "NVIDIA" not in vendor.upper() or not device:
        raise RuntimeError(
            f"NVIDIA EGL rendering is required; actual GL vendor={vendor!r}, renderer={device!r}"
        )
    if pixels.shape != (64, 64, 3) or nonzero_pixels == 0:
        raise RuntimeError("NVIDIA EGL did not produce nonempty 64x64 RGB pixels")
    graphics = {
        "gl_vendor": vendor, "gl_renderer": device, "shape": list(pixels.shape),
        "nonzero_pixels": nonzero_pixels, "render_ms": render_ms,
        "egl_vendor_config": os.environ["__EGL_VENDOR_LIBRARY_FILENAMES"],
    }
    Path(sys.argv[1]).write_text(json.dumps(graphics, indent=2) + "\n", encoding="utf-8")
    print("[PASS] actual NVIDIA EGL rendering:", json.dumps(graphics), flush=True)
finally:
    renderer.close()
"""
run([PYTHON, "-c", graphics_probe, SESSION / "graphics.json"], cwd=REPO)
graphics = json.loads((SESSION / "graphics.json").read_text())
print(json.dumps(graphics, indent=2))

run([
    PYTHON, "-m", "mjrl.colab", "check", "--repo", REPO,
    "--out", SESSION / "runtime.json",
], cwd=REPO)
runtime = json.loads((SESSION / "runtime.json").read_text())
print(json.dumps(runtime, indent=2))
for package, expected in CORE_VERSIONS.items():
    actual = runtime["versions"][package]
    if actual != expected:
        raise RuntimeError(f"Core version restore failed: {package} expected {expected}, got {actual}")
run([PYTHON, "scripts/train.py", "--list"], cwd=REPO)

# Detect an older replay CLI before attempting to train anything.
play_help = subprocess.check_output(
    [str(PYTHON), "scripts/play.py", "--help"], cwd=REPO, text=True,
)
required_video_flags = ("--video", "--video-fps", "--video-width", "--video-height")
if any(flag not in play_help for flag in required_video_flags):
    raise RuntimeError("This source revision lacks the notebook's replay video interface")

## 4. Optional Drive persistence

In [ ]:
PERSIST_ROOT = None
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    if not DRIVE_FOLDER or Path(DRIVE_FOLDER).is_absolute() or ".." in Path(DRIVE_FOLDER).parts:
        raise ValueError("DRIVE_FOLDER must be a relative folder under MyDrive")
    PERSIST_ROOT = Path("/content/drive/MyDrive") / DRIVE_FOLDER
    PERSIST_ROOT.mkdir(parents=True, exist_ok=True)
    print("Complete checkpoint snapshots will be saved under:", PERSIST_ROOT)
else:
    print("Drive is off. Results remain in /content until you download them.")

## 5. Watch the bundled pretrained dance; optionally restore your backup

The dance checkpoint is included in the repository. The video is a finite ten-second policy replay at the dance task's training physics rate (200 Hz), with training observation noise enabled. It demonstrates an existing checkpoint, not the untrained walking policy.

The optional upload restores a notebook backup into a **new** directory. It rejects traversal paths, symlinks, duplicate entries, oversized archives and replacement of existing runs. It selects a checkpoint only inside that restored run; compatibility is checked before training.

In [ ]:
from IPython.display import Image, Video, display

DEMO = REPO / "tasks/jumper/dance/out/example/model_2300.pt"
if not DEMO.is_file():
    raise FileNotFoundError(f"Bundled pretrained demo checkpoint is missing: {DEMO}")
DEMO_VIDEO = SESSION / "pretrained-dance.mp4"
run([
    PYTHON, "scripts/play.py", "--task", "jumper.dance", "--model", MODEL,
    "--checkpoint", DEMO, "--backend", "warp", "--device", "cuda:0",
    "--num_envs", "1", "--headless", "--steps", "500", "--physics-hz", "200",
    "--video", DEMO_VIDEO, "--video-fps", "30",
    "--video-width", "640", "--video-height", "480", "--strip-visual", "off",
], cwd=REPO)
if not DEMO_VIDEO.is_file() or DEMO_VIDEO.stat().st_size == 0:
    raise RuntimeError("Replay did not produce the demo MP4")
display(Video(str(DEMO_VIDEO), embed=True))

In [ ]:
if UPLOAD_BACKUP:
    from google.colab import files
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Upload exactly one notebook backup ZIP")
    original_name, payload = next(iter(uploaded.items()))
    archive = SESSION / "uploaded-backup.zip"
    archive.write_bytes(payload)
    restored = SESSION / "restored"
    restore_info = SESSION / "restore.json"
    run([
        PYTHON, "-m", "mjrl.colab", "restore", "--archive", archive,
        "--destination", restored, "--out", restore_info,
    ], cwd=REPO)
    RESUME_CHECKPOINT = json.loads(restore_info.read_text())["checkpoint"]
    print("Explicit restored resume checkpoint:", RESUME_CHECKPOINT)
else:
    print("No ZIP upload requested.")
if RESUME_CHECKPOINT:
    if not Path(RESUME_CHECKPOINT).is_file():
        raise FileNotFoundError(f"Resume checkpoint does not exist: {RESUME_CHECKPOINT}")
    print("Selected resume checkpoint:", RESUME_CHECKPOINT)

## 6. Five-iteration GPU smoke test

This uses 256 environments, `warp` and `cuda:0` explicitly. It must produce a new log directory and a complete checkpoint. If the assigned GPU runs out of memory, stop and reduce the configured batch for a new training session; this cell does not disguise an error as CPU success.

The smoke run is separate from the chosen training run and is never implicitly selected for resume.

In [ ]:
SMOKE_MANIFEST = SESSION / "smoke.json"
run([
    PYTHON, "-m", "mjrl.colab", "train", "--repo", REPO,
    "--task", "jumper.tripod", "--model", MODEL, "--num-envs", "256",
    "--iterations", "5", "--manifest", SMOKE_MANIFEST,
], cwd=REPO)
smoke = json.loads(SMOKE_MANIFEST.read_text())
if smoke["status"] != "finished":
    raise RuntimeError(f"Smoke test failed: {smoke}")
print("GPU smoke test completed:", smoke["run_directory"])

## 7. Train the selected task

`ITERATIONS` is the number of updates **this run performs**; on resume these are additional updates. The default 500 is deliberately modest and is not a convergence criterion. The repository's policies may require thousands of updates. Training with 256 environments changes PPO batch size from the original large GPU runs; matching their learning curves requires separate measurement.

Every complete new checkpoint is CRC-checked, hashed and recorded. With Drive on, the checkpoint and its manifest are copied to a temporary folder, then published together while training continues. A failed subprocess or interrupted copy remains an error, with a local failure manifest for diagnosis. Saved checkpoints remain usable even if later training fails.

Keep the same task, model, environment count and source/runtime versions when resuming. A bundled example checkpoint has no notebook run manifest and is suitable for replay, not implicit notebook resume.

In [ ]:
EXPORT_SUCCESS = False
EXPORTED_CHECKPOINT = ""
SIMULATION_CHECKPOINT = ""
MEASUREMENT_CHECKPOINT = ""
TRAIN_MANIFEST = SESSION / "training.json"
train_command = [
    PYTHON, "-m", "mjrl.colab", "train", "--repo", REPO,
    "--task", TASK, "--model", MODEL, "--num-envs", str(NUM_ENVS),
    "--iterations", str(ITERATIONS), "--manifest", TRAIN_MANIFEST,
]
if PERSIST_ROOT is not None:
    train_command += ["--drive-root", PERSIST_ROOT]
if RESUME_CHECKPOINT:
    train_command += ["--resume", Path(RESUME_CHECKPOINT)]
run(train_command, cwd=REPO)
training = json.loads(TRAIN_MANIFEST.read_text())
if training["status"] != "finished":
    raise RuntimeError(f"Training did not finish: {training}")
RUN_DIR = Path(training["run_directory"])
CHECKPOINT_PATH = Path(training["checkpoint"])
print("This run:", RUN_DIR)
print("Selected checkpoint:", CHECKPOINT_PATH)

# A separate attachment directory per run prevents a failed rerun from packaging old curves.
import shutil

ARTIFACTS = SESSION / "artifacts" / training["operation"]
ARTIFACTS.mkdir(parents=True, exist_ok=True)
for name in ("runtime.json", "graphics.json"):
    shutil.copy2(SESSION / name, ARTIFACTS / name)
print("Run attachments:", ARTIFACTS)

## 7.1. Inspect training curves

The TensorBoard scalar events from this exact run are exported to full JSON and CSV, then plotted inside the notebook. Inspect reward, episode length and losses to assess progress. A completed update count does not establish a useful gait. The numerical curve files remain in the backup even if a later preview fails.

Non-finite scalar samples are preserved as explicit NaN/Infinity strings with warning counts. Curves leave gaps at those samples; inspect the recorded flags rather than treating them as valid numbers.


In [ ]:
METRICS_JSON = ARTIFACTS / "metrics.json"
CURVES_PNG = ARTIFACTS / "training-curves.png"
run([
    PYTHON, "-m", "mjrl.colab", "metrics", "--run", RUN_DIR,
    "--out", METRICS_JSON, "--plot", CURVES_PNG,
], cwd=REPO)
metrics = json.loads(METRICS_JSON.read_text())
if Path(metrics["run"]).resolve() != RUN_DIR.resolve():
    raise RuntimeError("Training curves belong to a different run")
print("Full scalar history JSON:", METRICS_JSON)
print("Full scalar history CSV:", METRICS_JSON.with_suffix(".csv"))
display(Image(filename=str(CURVES_PNG)))


## 8. Simulate your policy and record its telemetry

The replay form sets the simulated duration, recording rate, even image dimensions, scene and camera distance. The task's actual control frequency determines the step count; recording FPS does not change physics or inference rates. **task default** retains the training task's world. Observation noise stays enabled. Replay runs real MuJoCo Warp physics and policy inference, then displays a finite MP4; Colab does not open a desktop keyboard/gamepad window.

With **RECORD_JOINTS** on, the existing `--measure` recorder writes one sample per control step. The CSV carries simulation time in seconds, absolute joint position in radians, joint velocity in radians/second and applied actuator torque in N·m, with separate joint/actuator names. Contact-enabled tasks may also record foot force in newtons. The resulting plot and CSV are included in the backup; they are simulation measurements, not hardware measurements.

To preview another scene after training, change only the SIM_* form settings, RECORD_JOINTS or DOWNLOAD_FILES, rerun the settings cell, then rerun the replay cell below. You do not need to repeat setup, the smoke test or training. Changing TASK or MODEL requires a new training run. Each replay keeps its earlier video and archives previous telemetry/settings under replay-history; the new backup contains the current recording and measurements. Rerun the download or backup cell to save the new result.


In [ ]:
# Replay settings can change without retraining; the policy task/model cannot.
recorded_config = training["config"]
if TASK != recorded_config["task"] or MODEL != recorded_config["model"]:
    raise ValueError(
        "The selected checkpoint belongs to a different task/model. "
        "Change only SIM_* or RECORD_JOINTS for replay; train a new run to change TASK/MODEL."
    )

# Preserve previous telemetry without putting it beside the new recording in the ZIP.
previous_outputs = [(CHECKPOINT_PATH.parent / "measure", "checkpoint-measure")]
previous_outputs += [(ARTIFACTS / name, name) for name in (
    "measure.csv", "measure.png", "simulation-settings.json",
)]
REPLAY_HISTORY = SESSION / "replay-history" / uuid.uuid4().hex
for previous, name in previous_outputs:
    if previous.is_symlink():
        raise RuntimeError(f"Replay refuses a symlink at its generated output: {previous}")
    if previous.exists():
        REPLAY_HISTORY.mkdir(parents=True, exist_ok=True)
        shutil.move(str(previous), str(REPLAY_HISTORY / name))

rates_file = SESSION / "rates.json"
run([
    PYTHON, "-m", "mjrl.colab", "rates", "--repo", REPO,
    "--task", TASK, "--model", MODEL, "--out", rates_file,
], cwd=REPO)
rates = json.loads(rates_file.read_text())
SIM_STEPS = math.ceil(SIM_SECONDS * rates["control_hz"])
print("Replay rates:", rates, "control steps:", SIM_STEPS)
SIMULATION_CHECKPOINT = ""
MEASUREMENT_CHECKPOINT = ""
TRAIN_VIDEO = SESSION / f"trained-policy-{training['operation']}-{uuid.uuid4().hex[:8]}.mp4"
replay_command = [
    PYTHON, "scripts/play.py", "--task", TASK, "--model", MODEL,
    "--checkpoint", CHECKPOINT_PATH, "--backend", "warp", "--device", "cuda:0",
    "--num_envs", "1", "--headless", "--steps", str(SIM_STEPS),
    "--physics-hz", str(rates["physics_hz"]), "--video", TRAIN_VIDEO,
    "--video-fps", str(SIM_FPS), "--video-width", str(SIM_WIDTH),
    "--video-height", str(SIM_HEIGHT), "--strip-visual", "off",
]
if SIM_SCENE != "task default":
    replay_command += ["--scene", SIM_SCENE]
if SIM_CAMERA_DISTANCE > 0:
    replay_command += ["--video-distance", str(SIM_CAMERA_DISTANCE)]
if RECORD_JOINTS:
    replay_command += ["--measure"]
run(replay_command, cwd=REPO)
if not TRAIN_VIDEO.is_file() or TRAIN_VIDEO.stat().st_size == 0:
    raise RuntimeError("Replay did not produce the trained MP4")
SIMULATION_CHECKPOINT = str(CHECKPOINT_PATH)
simulation_settings = {
    "checkpoint": str(CHECKPOINT_PATH), "run": str(RUN_DIR), "scene": SIM_SCENE,
    "physics_hz": rates["physics_hz"], "control_hz": rates["control_hz"],
    "requested_seconds": SIM_SECONDS, "simulated_seconds": SIM_STEPS / rates["control_hz"],
    "control_steps": SIM_STEPS, "fps": SIM_FPS, "width": SIM_WIDTH, "height": SIM_HEIGHT,
    "camera_distance": SIM_CAMERA_DISTANCE, "observation_noise": True,
    "record_joints": RECORD_JOINTS,
}
(ARTIFACTS / "simulation-settings.json").write_text(
    json.dumps(simulation_settings, indent=2) + "\n", encoding="utf-8",
)
display(Video(str(TRAIN_VIDEO), embed=True))
if RECORD_JOINTS:
    measurement_directory = CHECKPOINT_PATH.parent / "measure"
    for name in ("measure.csv", "measure.png"):
        source = measurement_directory / name
        if not source.is_file() or source.stat().st_size == 0:
            raise RuntimeError(f"Requested joint recording is missing: {source}")
        shutil.copy2(source, ARTIFACTS / name)
    MEASUREMENT_CHECKPOINT = str(CHECKPOINT_PATH)
    print("Joint telemetry CSV:", ARTIFACTS / "measure.csv")
    display(Image(filename=str(ARTIFACTS / "measure.png")))

In [ ]:
from google.colab import files

if DOWNLOAD_FILES:
    files.download(str(DEMO_VIDEO))
    files.download(str(TRAIN_VIDEO))
    if MEASUREMENT_CHECKPOINT == str(CHECKPOINT_PATH):
        files.download(str(ARTIFACTS / "measure.csv"))
else:
    print("Automatic downloads are off. Videos and telemetry remain in the session and backup.")

## 9. Export ONNX and download a portable backup

Export uses the same repository entry point and explicitly selected checkpoint. Its normal contract and ONNX checks run; `--no-video` avoids separately rendering a task's full motion clip. Repeating export writes a new directory. No RKNN conversion, board bundle or physical deployment happens here.

The ZIP includes the selected run's complete checkpoints, parameters/log files and provenance; runtime/GPU rendering reports; training-curve JSON, CSV and image; simulation settings and joint telemetry. Successful ONNX and MP4 results are included only when they belong to the selected checkpoint. Restore the ZIP through section 5 in a later session, choose its recorded source commit and core versions, and continue with the explicit restored checkpoint.

If curves, rendering or export fails, run the backup cell to preserve complete saved checkpoints and available attachments. With **DOWNLOAD_FILES** on, the notebook downloads videos, joint CSV and final ZIP. With it off, download the printed files manually before the runtime ends. Optional Drive snapshots protect checkpoint saves during training; a final ZIP alone cannot protect an interrupted session.

In [ ]:
EXPORT_SUCCESS = False
EXPORTED_CHECKPOINT = ""
EXPORT_DIR = SESSION / f"onnx-export-{uuid.uuid4().hex[:8]}"
run([
    PYTHON, "scripts/export.py", "--task", TASK, "--model", MODEL,
    "--checkpoint", CHECKPOINT_PATH, "--backend", "warp", "--device", "cuda:0",
    "--num_envs", "1", "--no-video", "--out", EXPORT_DIR,
], cwd=REPO)
if not (EXPORT_DIR / "actor.onnx").is_file() or not (EXPORT_DIR / "layout.json").is_file():
    raise RuntimeError("ONNX export did not produce its policy and contract")
print("ONNX policy and contract:", EXPORT_DIR)
EXPORTED_CHECKPOINT = str(CHECKPOINT_PATH)
EXPORT_SUCCESS = True
(ARTIFACTS / "export-settings.json").write_text(json.dumps({
    "checkpoint": EXPORTED_CHECKPOINT, "directory": str(EXPORT_DIR), "run": str(RUN_DIR),
}, indent=2) + "\n", encoding="utf-8")

In [ ]:
import shutil

from google.colab import files

# The manifest selects this run even when curves, replay or export failed later.
training = json.loads(TRAIN_MANIFEST.read_text())
if not training.get("run_directory") or not training.get("checkpoints"):
    raise RuntimeError("No complete training checkpoint is available to back up")
RUN_DIR = Path(training["run_directory"])
BACKUP_CHECKPOINT = str(Path(training["checkpoint"]))
ARTIFACTS = SESSION / "artifacts" / training["operation"]
ARTIFACTS.mkdir(parents=True, exist_ok=True)
for name in ("runtime.json", "graphics.json"):
    shutil.copy2(SESSION / name, ARTIFACTS / name)
BACKUP_ZIP = SESSION / f"jumper-{TASK.replace('.', '-')}-{uuid.uuid4().hex[:8]}.zip"
backup_command = [
    PYTHON, "-m", "mjrl.colab", "backup", "--run", RUN_DIR, "--out", BACKUP_ZIP,
    "--extras", ARTIFACTS,
]
if (globals().get("EXPORT_SUCCESS", False)
        and globals().get("EXPORTED_CHECKPOINT") == BACKUP_CHECKPOINT):
    backup_command += ["--export", EXPORT_DIR]
if (globals().get("SIMULATION_CHECKPOINT") == BACKUP_CHECKPOINT
        and TRAIN_VIDEO.is_file() and TRAIN_VIDEO.stat().st_size):
    backup_command += ["--video", TRAIN_VIDEO]
run(backup_command, cwd=REPO)
print("Portable backup:", BACKUP_ZIP, "bytes:", BACKUP_ZIP.stat().st_size)
if DOWNLOAD_FILES:
    files.download(str(BACKUP_ZIP))
else:
    print("Download this ZIP before Colab discards the runtime:", BACKUP_ZIP)

## Interruption and troubleshooting

- **No CUDA / Warp kernel failure:** attach a GPU runtime, inspect the driver and wheel errors, and rerun setup. No automatic CPU fallback is enabled.
- **Out of memory:** use a smaller training batch in a new run. Resume requires the recorded batch; changing it is a new experiment.
- **Task or source mismatch:** restore the same source commit and core package versions, not merely a checkpoint with a similar filename.
- **Training failed:** inspect `training.json` and console output. Download the ZIP of any complete saved checkpoints, or resume from a published Drive snapshot's explicit `model_*.pt` after a matching setup.
- **A Colab runtime disconnects:** inspect Drive snapshot folders; folders named `.pending-*` were not published and should not be selected. Without Drive or a downloaded backup, `/content` results can be lost.
- **Render/export failed:** the training run and checkpoint manifest remain available. The backup cell accepts a run without successful video or ONNX output.

The notebook's static checks and local process/backup tests are distinct from an actual Colab GPU session. Do not infer that a particular Colab GPU, training outcome or robot has been validated from the existence of this notebook.